# Multi-Disease Prediction with Fairness Audit

This notebook predicts **two related health conditions at once** from `fairness_auditing_healthcare_data.csv`:

1. **Glycemic status** (`Glycemic_Status_Encoded`) — 0 = Normal, 1 = Prediabetic, 2 = Diabetic
2. **BMI / metabolic status** (`BMI_Status_Encoded`) — 0 = Underweight, 1 = Normal, 2 = Overweight, 3 = Obese

**Important finding from exploring the data first:** symptom columns (frequent urination, thirst, fatigue, blurred vision) and demographic columns have ~0 correlation with these labels — the labels were generated purely by bucketing `HbA1c`/`Fasting_Blood_Glucose` and `BMI`. A model using only symptoms/demographics performs no better than guessing the majority class. The legitimate multi-disease model below uses the full clinical feature set (labs + symptoms + demographics), which is standard practice — HbA1c/glucose and BMI *are* the diagnostic inputs for these conditions in real medicine.

The second half of the notebook audits whether this model's accuracy holds up equally across gender, insurance, and socioeconomic groups — the actual point of a fairness-auditing exercise.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

df = pd.read_csv('fairness_auditing_healthcare_data.csv')
df.head()

,Age,HbA1c_mmol_mol,Total_Cholesterol_mg_dL,BMI (kg meter square),Glycemic_Status_Encoded,BMI_Status_Encoded,Symptom Frequent Urination,Symptom Increased Thirst,Symptom Fatigue,Symptom Blurred Vision,Socioeconomic_Status_Encoded,Insurance_Encoded,Gender_Male,Hospital Type_Government,Hospital Type_Private,Fasting_Blood_Glucose__mg_dl
0,0.441873,-0.264851,0.847229,-0.301979,1,1,0,1,1,0,0,1,0,1,0,-0.610669
1,-0.750094,-1.138065,0.731933,0.450902,1,2,0,0,0,0,0,0,1,0,1,-0.160981
2,-1.311020,1.770864,1.250764,0.827343,0,2,0,1,1,0,0,0,1,0,1,-1.188209
3,-0.890326,-0.730922,1.250764,0.262682,1,2,0,1,1,0,0,1,0,1,0,-0.570990
4,0.091294,-0.264851,1.250764,1.580223,0,3,1,0,0,1,2,0,0,1,0,-1.291814


## 1. Define the two disease targets and split data

In [2]:
targets = ['Glycemic_Status_Encoded', 'BMI_Status_Encoded']

X = df.drop(columns=targets)
y = df[targets]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(X_train.shape, X_test.shape)

(80000, 14) (20000, 14)


## 2. Train one multi-output model that predicts both diseases at once

`MultiOutputClassifier` fits one classifier per target internally, but you call `.fit()` / `.predict()` once, and get both predictions back together — this is the actual "multi-disease" model.

In [3]:
base_model = LogisticRegression(max_iter=1000)
multi_model = MultiOutputClassifier(base_model)
multi_model.fit(X_train, y_train)

y_pred = multi_model.predict(X_test)
y_pred_df = pd.DataFrame(y_pred, columns=targets, index=X_test.index)

## 3. Evaluate each disease target separately

In [4]:
for i, col in enumerate(targets):
    acc = accuracy_score(y_test[col], y_pred_df[col])
    f1 = f1_score(y_test[col], y_pred_df[col], average='macro')
    print(f"=== {col} ===")
    print(f"Accuracy: {acc:.3f} | Macro-F1: {f1:.3f}")
    print(classification_report(y_test[col], y_pred_df[col]))
    print()

=== Glycemic_Status_Encoded ===
Accuracy: 0.999 | Macro-F1: 0.999
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      5775
           1       1.00      1.00      1.00      5628
           2       1.00      1.00      1.00      8597

    accuracy                           1.00     20000
   macro avg       1.00      1.00      1.00     20000
weighted avg       1.00      1.00      1.00     20000


=== BMI_Status_Encoded ===
Accuracy: 0.941 | Macro-F1: 0.947
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2788
           1       0.94      0.96      0.95      7817
           2       0.89      0.92      0.91      6178
           3       0.99      0.88      0.93      3217

    accuracy                           0.94     20000
   macro avg       0.96      0.94      0.95     20000
weighted avg       0.94      0.94      0.94     20000




## 4. Fairness audit — does accuracy hold up across groups?

For each sensitive attribute, compute per-group accuracy on **both** disease predictions. Large gaps between groups indicate the model performs unevenly for that subgroup — the core question a fairness audit is meant to answer.

In [5]:
audit_df = X_test.copy()
for col in targets:
    audit_df[f'{col}_true'] = y_test[col].values
    audit_df[f'{col}_pred'] = y_pred_df[col].values
    audit_df[f'{col}_correct'] = (audit_df[f'{col}_true'] == audit_df[f'{col}_pred']).astype(int)

sensitive_attrs = ['Gender_Male', 'Insurance_Encoded', 'Socioeconomic_Status_Encoded',
                    'Hospital Type_Government', 'Hospital Type_Private']

for attr in sensitive_attrs:
    print(f'--- Accuracy by {attr} ---')
    print(audit_df.groupby(attr)[[f'{t}_correct' for t in targets]].mean().round(3))
    print()

--- Accuracy by Gender_Male ---
             Glycemic_Status_Encoded_correct  BMI_Status_Encoded_correct
Gender_Male                                                             
0                                      0.999                       0.940
1                                      0.999                       0.942

--- Accuracy by Insurance_Encoded ---
                   Glycemic_Status_Encoded_correct  BMI_Status_Encoded_correct
Insurance_Encoded                                                             
0                                            0.999                       0.940
1                                            0.998                       0.942

--- Accuracy by Socioeconomic_Status_Encoded ---
                              Glycemic_Status_Encoded_correct  \
Socioeconomic_Status_Encoded                                    
0                                                       0.999   
1                                                       0.999   
2          

## 5. Deeper fairness check — false negative rate for "Diabetic" / "Obese" by group

Overall accuracy can hide disparities that show up only in the highest-risk class (missing a diabetic or obese patient is a worse error than misclassifying a normal one). This checks the miss rate specifically for the highest-risk class of each target, by group.

In [6]:
def false_negative_rate(true, pred, positive_class):
    positives = true == positive_class
    if positives.sum() == 0:
        return np.nan
    missed = (true == positive_class) & (pred != positive_class)
    return missed.sum() / positives.sum()

high_risk_class = {'Glycemic_Status_Encoded': 2, 'BMI_Status_Encoded': 3}

for attr in sensitive_attrs:
    print(f'--- FNR for high-risk class, by {attr} ---')
    for group, gdf in audit_df.groupby(attr):
        row = {}
        for col, cls in high_risk_class.items():
            row[col] = round(
                false_negative_rate(gdf[f'{col}_true'], gdf[f'{col}_pred'], cls), 3
            )
        print(f'  {attr}={group}: {row}')
    print()

--- FNR for high-risk class, by Gender_Male ---
  Gender_Male=0: {'Glycemic_Status_Encoded': np.float64(0.002), 'BMI_Status_Encoded': np.float64(0.122)}
  Gender_Male=1: {'Glycemic_Status_Encoded': np.float64(0.001), 'BMI_Status_Encoded': np.float64(0.126)}

--- FNR for high-risk class, by Insurance_Encoded ---
  Insurance_Encoded=0: {'Glycemic_Status_Encoded': np.float64(0.001), 'BMI_Status_Encoded': np.float64(0.128)}
  Insurance_Encoded=1: {'Glycemic_Status_Encoded': np.float64(0.002), 'BMI_Status_Encoded': np.float64(0.122)}

--- FNR for high-risk class, by Socioeconomic_Status_Encoded ---
  Socioeconomic_Status_Encoded=0: {'Glycemic_Status_Encoded': np.float64(0.002), 'BMI_Status_Encoded': np.float64(0.127)}
  Socioeconomic_Status_Encoded=1: {'Glycemic_Status_Encoded': np.float64(0.001), 'BMI_Status_Encoded': np.float64(0.123)}
  Socioeconomic_Status_Encoded=2: {'Glycemic_Status_Encoded': np.float64(0.001), 'BMI_Status_Encoded': np.float64(0.121)}

--- FNR for high-risk class, by 

## 6. Save predictions for reporting

In [7]:
results = X_test.copy()
for col in targets:
    results[f'{col}_Actual'] = y_test[col].values
    results[f'{col}_Predicted'] = y_pred_df[col].values

results.to_csv('multi_disease_predictions.csv', index=False)
print("Saved multi_disease_predictions.csv")

Saved multi_disease_predictions.csv
